# 07 — Compare experiments and limitations

### How to read this notebook
Run cells top to bottom. Each Markdown cell states the question and expected result **before** the code. The printed and plotted numbers are measurements, not claims made in advance. The shared functions live in `src/experiment.py`. Open Jupyter from the project root so imports work.

## Goal

Read the CSVs saved by the experimental notebooks. Compare full-data accuracy and conflicting-feature reliance. The reliance diagnostic uses **out-of-distribution conflicts**, so it should not be interpreted as a uniquely identified in-distribution causal contribution.

The toy dataset contains four input prototypes. Training and evaluation on repeated versions of those prototypes is not evidence of generalization to unseen structures.

This notebook compares the single-seed demonstrations from Notebooks 03–05. For the complete five-seed sweep and counterfactual metrics, use Notebooks 08 and 09.

In [1]:
from src.experiment import *
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
print('PyTorch:', torch.__version__)

PyTorch: 2.14.1+cpu


In [2]:
import csv
files=['results/03_ordinary.csv','results/04_inverse.csv','results/05_compensated.csv']
rows=[]
for filename in files:
    path=Path(filename)
    if not path.exists():
        print('Not yet available:',filename,'— run the corresponding notebook first.')
        continue
    with path.open() as f:
        rows.extend(list(csv.DictReader(f)))
for r in rows:
    print(f"{r['experiment']:>24} alpha={r['alpha']:>4} accuracy={float(r['accuracy']):.3f} complex_reliance={float(r['complex_reliance']):.3f}")

                ordinary alpha= 0.1 accuracy=1.000 complex_reliance=0.000
                ordinary alpha= 0.5 accuracy=1.000 complex_reliance=0.000
                ordinary alpha=   1 accuracy=1.000 complex_reliance=0.000
                ordinary alpha=   2 accuracy=1.000 complex_reliance=1.000
                ordinary alpha=   5 accuracy=1.000 complex_reliance=1.000
                ordinary alpha=  10 accuracy=1.000 complex_reliance=1.000
       inverse_reference alpha=   1 accuracy=1.000 complex_reliance=0.000
          inverse_scaled alpha= 5.0 accuracy=1.000 complex_reliance=1.000
   compensated_reference alpha=   1 accuracy=1.000 complex_reliance=0.000
      compensated_scaled alpha=   5 accuracy=1.000 complex_reliance=0.000


## How to interpret possible outcomes

- **Ordinary scaling differs, inverse init does not:** the effect depends on something changed by ordinary scaling but removed by matching the initial function; it does not uniquely identify ReLU gates.
- **Inverse init still differs, compensated SGD matches:** consistent with effective plain-SGD parameterization differences.
- **No reliance differences:** this setup provides no evidence for a scale-driven reliance reversal; inspect seed variation and whether the metric is informative.
- **Compensated SGD fails to match:** investigate code or violated mathematical assumptions before interpreting results.

## Next checks

Repeat across seeds, report variability, and pre-register how reliance is measured. Avoid claiming to disprove simplicity bias broadly. An out-of-distribution conflict diagnostic measures a particular behavioral preference, not all possible notions of feature reliance.